# Despliegue

- Cargamos el modelo
- Cargamos los datos futuros
- Preparar los datos futuros
- Aplicamos el modelo para la predicción

In [ ]:
#Cargamos librerías principales

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import pickle
filename = 'modelo-classNN.pkl'

# Cargamos el contenido del archivo pickle, que es una lista de 4 elementos
loaded_content = pickle.load(open(filename, 'rb'))
modelo, label_encoder, variables, min_max_scaler = loaded_content

Tipo de modelo: <class 'sklearn.neural_network._multilayer_perceptron.MLPClassifier'>
Tipo de label_encoder: <class 'sklearn.preprocessing._label.LabelEncoder'>
Tipo de variables: <class 'numpy.ndarray'>
Tipo de min_max_scaler: <class 'sklearn.preprocessing._data.MinMaxScaler'>


In [ ]:
#Cargamos los datos futuros
#data = pd.read_csv("DatosFuturosP4.csv")
#data.head()

,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,attack_detected
0,5,450.5,AES,0.65,1,Chrome,0
1,2,120.3,NaN,0.12,0,Firefox,0
2,8,890.7,DES,0.88,3,Edge,1
3,4,550.2,AES,0.45,1,Safari,0
4,6,720.9,AES,0.72,2,Chrome,1


In [ ]:
#Interfaz gráfica
#Se crea interfaz gráfica con streamlit para captura de los datos

import streamlit as st

st.title('Predicción de Riesgo de Seguridad')

# Inputs para el modelo
login_attempts = st.number_input('Número de intentos de inicio de sesión', min_value=0, value=5, step=1)
session_duration = st.number_input('Duración de la sesión (minutos)', min_value=0.0, value=450.5, step=0.1)
ip_reputation_score = st.number_input('Puntuación de reputación IP (0-1)', min_value=0.0, max_value=1.0, value=0.65, step=0.01)
failed_logins = st.number_input('Número de inicios de sesión fallidos', min_value=0, value=1, step=1)
encryption_used = st.selectbox('Cifrado utilizado', ['AES', 'DES', 'None'])
browser_type = st.selectbox('Tipo de navegador', ['Chrome', 'Firefox', 'Edge', 'Safari', 'Unknown'])


#Dataframe
datos = [[login_attempts, session_duration, ip_reputation_score, failed_logins, encryption_used, browser_type]]
data = pd.DataFrame(datos, columns=['login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins', 'encryption_used', 'browser_type']) #Dataframe con los mismos nombres de variables

In [ ]:
#Se realiza la preparación
data_preparada=data.copy()

# En despliegue drop_first= False
data_preparada = pd.get_dummies(data_preparada, columns=['encryption_used', 'browser_type'], drop_first=False, dtype=int)
data_preparada.head()

,login_attempts,session_duration,ip_reputation_score,failed_logins,attack_detected,encryption_used_AES,encryption_used_DES,browser_type_Chrome,browser_type_Edge,browser_type_Firefox,browser_type_Safari
0,5,450.5,0.65,1,0,1,0,1,0,0,0
1,2,120.3,0.12,0,0,0,0,0,0,1,0
2,8,890.7,0.88,3,1,0,1,0,1,0,0
3,4,550.2,0.45,1,0,1,0,0,0,0,1
4,6,720.9,0.72,2,1,1,0,1,0,0,0


In [ ]:
#Se adicionan las columnas faltantes
data_preparada=data_preparada.reindex(columns=variables,fill_value=0)
data_preparada.head()

,login_attempts,session_duration,ip_reputation_score,failed_logins,encryption_used_AES,encryption_used_DES,encryption_used_None,browser_type_Chrome,browser_type_Edge,browser_type_Firefox,browser_type_Safari,browser_type_Unknown
0,5,450.5,0.65,1,1,0,0,1,0,0,0,0
1,2,120.3,0.12,0,0,0,0,0,0,1,0,0
2,8,890.7,0.88,3,0,1,0,0,1,0,0,0
3,4,550.2,0.45,1,1,0,0,0,0,0,1,0
4,6,720.9,0.72,2,1,0,0,1,0,0,0,0


In [ ]:
# Identificamos las columnas numéricas que necesitan escalado (las que no son one-hot-encoded ni el objetivo)
numerical_cols = ['login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins']

# Aplicamos el scaler a todas las columnas numéricas de una vez
# Aseguramos que las columnas existen antes de intentar escalarlas

# Filtramos numerical_cols para incluir solo las que realmente están en data_preparada
existing_numerical_cols = [col for col in numerical_cols if col in data_preparada.columns]

if existing_numerical_cols:
    data_preparada[existing_numerical_cols] = min_max_scaler.transform(data_preparada[existing_numerical_cols])

data_preparada.head()

,login_attempts,session_duration,ip_reputation_score,failed_logins,encryption_used_AES,encryption_used_DES,encryption_used_None,browser_type_Chrome,browser_type_Edge,browser_type_Firefox,browser_type_Safari,browser_type_Unknown
0,0.333333,0.062588,0.702432,0.2,1,0,0,1,0,0,0,0
1,0.083333,0.016662,0.127471,0.0,0,0,0,0,0,1,0,0
2,0.583333,0.123813,0.951943,0.6,0,1,0,0,1,0,0,0
3,0.250000,0.076455,0.485465,0.2,1,0,0,0,0,0,1,0
4,0.416667,0.100196,0.778370,0.4,1,0,0,1,0,0,0,0


# Predicciones

In [ ]:
# Hacemos la predicción con la Red Neuronal (NN)
Y_pred = modelo.predict(data_preparada)
print(Y_pred)

[1 0 1 0 1]


In [ ]:
data['Prediccion']=Y_pred
data.head()

,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,attack_detected,Prediccion
0,5,450.5,AES,0.65,1,Chrome,0,1
1,2,120.3,NaN,0.12,0,Firefox,0,0
2,8,890.7,DES,0.88,3,Edge,1,1
3,4,550.2,AES,0.45,1,Safari,0,0
4,6,720.9,AES,0.72,2,Chrome,1,1


In [ ]:
#Predicciones finales
data

,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,attack_detected,Prediccion
0,5,450.5,AES,0.65,1,Chrome,0,1
1,2,120.3,NaN,0.12,0,Firefox,0,0
2,8,890.7,DES,0.88,3,Edge,1,1
3,4,550.2,AES,0.45,1,Safari,0,0
4,6,720.9,AES,0.72,2,Chrome,1,1


In [ ]:
# Recordar medida de error del modelo

st.warning("El modelo tiene un error del 20%")